- Aggs the yml and build files for each repo

In [1]:
import pandas as pd
import os

# === CONFIGURATION ===
base_dir = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31"
output_dir = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\Descriptive_Stat"

# === FILE PATHS ===
yml_csv = os.path.join(base_dir, "3.1_YML_List_ShallowC.csv")
yml_output_csv = os.path.join(output_dir, "3.1_YML_List_ShallowC_Aggregated.csv")

gradle_csv = os.path.join(base_dir, "3.2_Gradle_List_ShallowC.csv")
gradle_output_csv = os.path.join(output_dir, "3.2_Gradle_List_ShallowC_Aggregated.csv")

# === LOAD FILES ===
df_yml = pd.read_csv(yml_csv)
df_gradle = pd.read_csv(gradle_csv)

# === PREPROCESS YML COLUMNS (convert Yes/No → Boolean) ===
df_yml['Instru_T_Trigger'] = df_yml['Instru_T_Trigger'].map({'Yes': True, 'No': False})
df_yml['Instru_T_Device_Setup'] = df_yml['Instru_T_Device_Setup'].map({'Yes': True, 'No': False})
df_yml['instrumentation_test'] = df_yml['instrumentation_test'].astype(bool)
df_yml['unit_test'] = df_yml['unit_test'].astype(bool)

# === ONE-HOT COUNT FOR CI PLATFORM (YML) ===
df_yml['ci_platform'] = df_yml['ci_platform'].fillna('None').infer_objects(copy=False)
yml_platform_dummies = pd.get_dummies(df_yml['ci_platform']).groupby(df_yml['full_name']).sum()
yml_platform_dummies.columns = [f'yml_{col}' for col in yml_platform_dummies.columns]

# === AGGREGATE YML ===
df_yml_agg = df_yml.groupby('full_name').agg({
    'ci_platform': lambda x: ', '.join(sorted(set(filter(pd.notna, x)))),
    'test_type': lambda x: ', '.join(sorted(set(filter(pd.notna, x)))),
    'matched_keywords': lambda x: ', '.join(sorted(set(filter(pd.notna, x)))),
    'unit_test': 'any',
    'instrumentation_test': 'any',
    'Instru_T_Trigger': 'any',
    'Instru_T_Device_Setup': 'any',
    'full_name': 'count'
}).rename(columns={
    'unit_test': 'unit_test_ci',
    'instrumentation_test': 'instr_test_ci',
    'ci_platform': 'ci_platform_yml',
    'full_name': 'NBR_YAML'
}).reset_index()

# === Convert boolean back to Yes/No for output ===
# df_yml_agg['Instru_T_Trigger'] = df_yml_agg['Instru_T_Trigger'].map({True: 'Yes', False: 'No'})
# df_yml_agg['Instru_T_Device_Setup'] = df_yml_agg['Instru_T_Device_Setup'].map({True: 'Yes', False: 'No'})
# df_yml_agg['instr_test_ci'] = df_yml_agg['instr_test_ci'].map({True: 'Yes', False: 'No'})
# df_yml_agg['unit_test_ci'] = df_yml_agg['unit_test_ci'].map({True: 'Yes', False: 'No'})

# === MERGE PLATFORM COUNTS INTO YML AGGREGATE ===
df_yml_agg = df_yml_agg.merge(yml_platform_dummies, on='full_name', how='left')

# === EXPORT YML ===
df_yml_agg.to_csv(yml_output_csv, index=False)
print(f"✅ Aggregated YML file saved to: {yml_output_csv}")

# === ONE-HOT COUNT FOR CI PLATFORM (Gradle) ===
df_gradle['ci_platform_build'] = df_gradle['ci_platform_build'].fillna('None')
gradle_platform_dummies = pd.get_dummies(df_gradle['ci_platform_build']).groupby(df_gradle['full_name']).sum()
gradle_platform_dummies.columns = [f'build_{col}' for col in gradle_platform_dummies.columns]

# === AGGREGATE GRADLE ===
df_gradle_agg = df_gradle.groupby('full_name').agg({
    'has_build_unit_test': 'any',
    'has_build_instrumentation_test': 'any',
    'ci_platform_build': lambda x: ', '.join(sorted(set(filter(pd.notna, x)))),
    'full_name': 'count'
}).rename(columns={
    'has_build_unit_test': 'unit_test_build',
    'has_build_instrumentation_test': 'instr_test_build',
    'ci_platform_build': 'ci_platform_build',
    'full_name': 'NBR_GRADLE'
}).reset_index()

# === Convert booleans to Yes/No for Gradle too (optional) ===
# df_gradle_agg['unit_test_build'] = df_gradle_agg['unit_test_build'].map({True: 'Yes', False: 'No'})
# df_gradle_agg['instr_test_build'] = df_gradle_agg['instr_test_build'].map({True: 'Yes', False: 'No'})

# === MERGE PLATFORM COUNTS INTO GRADLE AGGREGATE ===
df_gradle_agg = df_gradle_agg.merge(gradle_platform_dummies, on='full_name', how='left')

# === EXPORT GRADLE ===
df_gradle_agg.to_csv(gradle_output_csv, index=False)
print(f"✅ Aggregated Gradle BUILD file saved to: {gradle_output_csv}")


✅ Aggregated YML file saved to: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\Descriptive_Stat\3.1_YML_List_ShallowC_Aggregated.csv
✅ Aggregated Gradle BUILD file saved to: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\Descriptive_Stat\3.2_Gradle_List_ShallowC_Aggregated.csv


API_Level per repo

In [2]:
import pandas as pd
import os

# === Input Paths ===
api_input_path = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\3.3_Project_List_API_Merged_Details.csv"
yml_input_path = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\Descriptive_Stat\3.1_YML_List_ShallowC_Aggregated.csv"
gradle_input_path = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\Descriptive_Stat\3.2_Gradle_List_ShallowC_Aggregated.csv"

# === Output Path ===
output_path = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\Descriptive_Stat\3.3_API_Levels.csv"

# === Load Data ===
df_api = pd.read_csv(api_input_path)
df_yml = pd.read_csv(yml_input_path)
df_gradle = pd.read_csv(gradle_input_path)

# === Ensure lowercase consistency ===
df_api['full_name'] = df_api['full_name'].str.lower()
df_yml['full_name'] = df_yml['full_name'].str.lower()
df_gradle['full_name'] = df_gradle['full_name'].str.lower()

# === Merge instr_test_build and instr_test_ci flags from both sources ===
df_flags = pd.merge(
    df_yml[['full_name', 'instr_test_ci']], 
    df_gradle[['full_name', 'instr_test_build']],
    on='full_name', 
    how='outer'
).fillna(False).infer_objects(copy=False)

# Ensure boolean type
df_flags['instr_test_ci'] = df_flags['instr_test_ci'].astype(bool)
df_flags['instr_test_build'] = df_flags['instr_test_build'].astype(bool)

# === Identify repos with either type of instrumentation test ===
df_flags['has_instr_test'] = df_flags['instr_test_ci'] | df_flags['instr_test_build']

# === Filter API-level rows to include only those repos with instrumentation tests ===
valid_full_names = df_flags[df_flags['has_instr_test']]['full_name'].unique()
df_api_filtered = df_api[df_api['full_name'].isin(valid_full_names)]

# === Pivot table: one column per API level ===
api_counts = pd.pivot_table(
    df_api_filtered,
    index='full_name',
    columns='api_level',
    aggfunc='size',
    fill_value=0
)

# === Rename columns with "api_" prefix ===
api_counts.columns = [f"api_{col}" for col in api_counts.columns]

# === Add total distinct API levels count per repo ===
api_counts['api_level_count'] = api_counts.sum(axis=1)

# === Save result ===
summary_df = api_counts.reset_index()
summary_df.to_csv(output_path, index=False)

print(f"✅ Filtered API summary saved to: {output_path}")


✅ Filtered API summary saved to: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\Descriptive_Stat\3.3_API_Levels.csv


C:\Users\gilla\AppData\Local\Temp\ipykernel_19260\3611373386.py:28: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(False).infer_objects(copy=False)


- creates the main list of repo
- adds language, Android_Keyword,nbr_of_manifest,Standard_Mannifest of each repo from previous step (searched urls)
- adds the required metasta of each repo

In [ ]:
import pandas as pd
import os
from datetime import datetime

# === CONFIGURATION ===
BASE_PATH = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31"
DESCRIPTIVE_PATH = os.path.join(BASE_PATH, "Descriptive_Stat")

INPUT_CSV = os.path.join(BASE_PATH, "Sorted_URL_List.csv")
OUTPUT_CSV = os.path.join(DESCRIPTIVE_PATH, "3.4_Total_Repos.csv")

LANGUAGE_CSV = os.path.join(BASE_PATH, "step4_ci_detection_output.csv")
REVIEW_STATUS_CSV = os.path.join(BASE_PATH, "Clone_Status.csv")
METADATA_CSV = os.path.join(BASE_PATH, "Project_Metadata.csv")

YML_FILE = os.path.join(DESCRIPTIVE_PATH, "3.1_YML_List_ShallowC_Aggregated.csv")
BUILD_FILE = os.path.join(DESCRIPTIVE_PATH, "3.2_Gradle_List_ShallowC_Aggregated.csv")


def clean_full_name(series):
    return (
        series.astype(str)
        .str.encode('ascii', 'ignore').str.decode('utf-8')  # remove non-ascii
        .str.replace(r'[\u200b\u200e\u202c\r\n\t]', '', regex=True)  # remove invisible chars
        .str.strip()
        .str.lower()
    )


# === LOAD MAIN REPO LIST ===
df = pd.read_csv(INPUT_CSV)
df['github_url'] = df['github_url'].astype(str)

# === EXTRACT COMPONENTS IN LOWERCASE ===
df['username'] = df['github_url'].apply(lambda x: x.split('/')[-2].lower())
#df['project_name'] = df['github_url'].apply(lambda x: x.split('/')[-1].lower().removesuffix('.git'))
df['project_name'] = df['github_url'].apply(lambda x: x.split('/')[-1].lower())
df['full_name'] = df['username'] + '.' + df['project_name']
df = df.reset_index()

# === MERGE: LANGUAGE + Android Info from step4 ===
df_lang = pd.read_csv(LANGUAGE_CSV)
df_lang['html_url'] = df_lang['html_url'].astype(str)
df = df.merge(
    df_lang[['html_url', 'language', 'Android_keyword', 'nbr_of_manifest', 'Standard_Manifest']],
    left_on='github_url',
    right_on='html_url',
    how='left'
)
df.drop(columns=['html_url'], inplace=True)

# === MERGE: REVIEW STATUS ===
df_review = pd.read_csv(REVIEW_STATUS_CSV)
df_review = df_review.rename(columns={'github_url': 'html_url'})
df_review['html_url'] = df_review['html_url'].astype(str)
df = df.merge(df_review[['html_url', 'clone_status']], left_on='github_url', right_on='html_url', how='left')
df.drop(columns=['html_url'], inplace=True)

# === MERGE: METADATA ===
df_meta = pd.read_csv(METADATA_CSV)
df_meta['html_url'] = df_meta['html_url'].astype(str)
meta_fields = [
    'html_url', 'created_at', 'size', 'stargazers_count', 'updated_at',
    'pushed_at', 'forks_count', 'contributors',
    'pull_requests', 'commits_GitAPI'
]
df = df.merge(df_meta[meta_fields], left_on='github_url', right_on='html_url', how='left')
df.drop(columns=['html_url'], inplace=True)

# === YML & BUILD Detection (case-insensitive match using lowercase full_name) ===
df_yml = pd.read_csv(YML_FILE)
df_build = pd.read_csv(BUILD_FILE)


df['full_name'] = clean_full_name(df['full_name'])
df_yml['full_name'] = clean_full_name(df_yml['full_name'])
df_build['full_name'] = clean_full_name(df_build['full_name'])


df_yml['full_name'] = df_yml['full_name'].str.lower()
df_build['full_name'] = df_build['full_name'].str.lower()

df['YML_Detected'] = df['full_name'].isin(df_yml['full_name']).map({True: "Yes", False: "No"})
df['Build_Detected'] = df['full_name'].isin(df_build['full_name']).map({True: "Yes", False: "No"})

# === CALCULATE AGE IN YEARS FROM August 01, 2025 ===
cutoff_date = pd.to_datetime("2025-08-01").tz_localize(None)
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce').dt.tz_localize(None)
df['repo_age_years'] = round((cutoff_date - df['created_at']).dt.days / 365.25,2)

# === FINAL COLUMN ORDER ===
final_cols = [
    'index', 'github_url', 'full_name', 'clone_status', 'language',
    'Android_keyword', 'nbr_of_manifest', 'Standard_Manifest',
    'created_at', 'repo_age_years', 'size', 'stargazers_count', 'updated_at', 'pushed_at',
    'forks_count', 'contributors', 'pull_requests', 'commits_GitAPI',
    'YML_Detected', 'Build_Detected'
]
df_clean = df[final_cols]

# === SAVE TO CSV ===
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
df_clean.to_csv(OUTPUT_CSV, index=False)

print(f"✅ Final enriched repo list with YML/Build detection saved to: {OUTPUT_CSV}")


✅ Final enriched repo list with YML/Build detection saved to: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\Descriptive_Stat\3.4_Total_Repos.csv


- Append the ci_platform and test related of each repo
- Create General CI Platform and Unit / instr testing for each repo

In [4]:
import pandas as pd
import os

# === FILE PATHS ===
base_dir = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\Descriptive_Stat"
main_file = os.path.join(base_dir, "3.4_Total_Repos.csv")
yml_file = os.path.join(base_dir, "3.1_YML_List_ShallowC_Aggregated.csv")
gradle_file = os.path.join(base_dir, "3.2_Gradle_List_ShallowC_Aggregated.csv")
api_file = os.path.join(base_dir, "3.3_API_Levels.csv")
output_file = os.path.join(base_dir, "3.4_Total_Repos_With_Metadata.csv")

# === LOAD FILES ===
df_main = pd.read_csv(main_file)
# ✅ KEEP ONLY REPOS WITH Clone_Status == 'yes'
df_main = df_main[df_main['clone_status'].astype(str).str.lower() == 'yes'].copy()
df_yml = pd.read_csv(yml_file)
df_gradle = pd.read_csv(gradle_file)
df_api = pd.read_csv(api_file)

# === NORMALIZE 'full_name' for safe joins ===
for df in [df_main, df_yml, df_gradle, df_api]:
    df['full_name'] = df['full_name'].astype(str).str.strip().str.lower()

# === MERGE ON 'full_name' ===
df_merged = df_main.merge(df_yml, on='full_name', how='left', suffixes=('', '_yml'))
df_merged = df_merged.merge(df_gradle, on='full_name', how='left', suffixes=('', '_gradle'))
df_merged = df_merged.merge(df_api, on='full_name', how='left')

# === DROP unused columns if they exist ===
columns_to_drop = ['ci_platform_build', 'build_Other']
df_merged.drop(columns=[col for col in columns_to_drop if col in df_merged.columns], inplace=True)

# === ADD: Combined Test Logic ===
df_merged['Unit_Test(CI or build)'] = (df_merged['unit_test_ci'] == True) | (df_merged['unit_test_build'] == True)
df_merged['Instr_Test(CI or build)'] = (df_merged['instr_test_ci'] == True) | (df_merged['instr_test_build'] == True)

# === SAVE ===
df_merged.to_csv(output_file, index=False)
print(f"✅ Final enriched repo list with test info saved to: {output_file}")


✅ Final enriched repo list with test info saved to: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\Descriptive_Stat\3.4_Total_Repos_With_Metadata.csv
